# 03. Splitting Text: Chunk Size, Overlap, and Boundary Safety

> **Promise:** By the end of this notebook, you will master document chunking, understand the trade-offs of chunk size versus overlap, and prevent information loss when cutting text.

### Learning Objectives
- Explain why chunking is required for search precision and model context limits.
- Master RecursiveCharacterTextSplitter and its delimiter hierarchy (\n\n, \n, space, empty string).
- Demonstrate a real split failure that cuts an answer in half and resolve it using chunk overlap.

### Prerequisites
- Notebook 02 (Loading Documents)


## 1. Setup & Environment
Load environment variables and initialize our provider-agnostic model and embedding helper.

In [1]:
import os, sys
from pathlib import Path

# Add project root to path for shared utilities
sys.path.insert(0, str(Path.cwd().parent.parent if Path.cwd().name == "notebooks" else Path.cwd()))
from course.utils.helpers import get_data_dir
from langchain_community.document_loaders import TextLoader

data_dir = get_data_dir()
handbook_doc = TextLoader(str(data_dir / "company_handbook.txt"), encoding="utf-8").load()[0]
print(f"Loaded handbook: {len(handbook_doc.page_content)} characters.")

C:\Users\Rahat\AppData\Local\Temp\ipykernel_8860\2111682962.py:7: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import TextLoader


Loaded handbook: 4163 characters.


## 2. Concept & Intuition

### Why Can't We Just Search Full Documents?

1. **Context Limits & Cost:** Sending a 50-page document to an LLM for a 1-sentence answer wastes tokens and causes high latency.
2. **Search Precision:** If an entire 50-page document is represented by a single coordinate vector, specific detailed facts (like a meal expense limit) get drowned out in the average.
3. **Chunking** cuts documents into bite-sized semantic passages so search finds the exact needle in the haystack.

#### Chunk Size vs. Chunk Overlap
- **Chunk Size:** The maximum number of characters (or tokens) per piece.
  - *Too small (e.g. 50 chars):* Cuts sentences in half; loses context.
  - *Too large (e.g. 3000 chars):* Dilutes search precision; wastes tokens.
- **Chunk Overlap:** The number of characters shared between adjacent chunks. It creates a sliding window that prevents facts at the boundary from being severed.

```
NO OVERLAP (Risk of cut sentence):
[ Chunk 1: Employees are reimbursed up to $150 ] [ Chunk 2: per day for meals without VP approval ]
                                                ▲
                                          Broken Meaning!

WITH OVERLAP (Boundary Preserved):
[ Chunk 1: Employees are reimbursed up to $150 per day for meals... ]
             [ Chunk 2: ...up to $150 per day for meals without VP approval. ]
```

## 3. Step-by-Step Hands-On Build

### Step 1: The Recursive Splitting Hierarchy
The gold standard text splitter is `RecursiveCharacterTextSplitter`. It attempts to split by paragraphs (`\n\n`) first. If chunks are still too large, it tries lines (`\n`), then spaces (` `), and only as a last resort splits by characters.

In [2]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=400,
    chunk_overlap=50,
    separators=["\n\n", "\n", " ", ""]
)

chunks = splitter.split_documents([handbook_doc])
print(f"Document split into {len(chunks)} chunks.")
print("Chunk #1 preview:\n", chunks[0].page_content)
print("\nChunk #2 preview:\n", chunks[1].page_content)

Document split into 13 chunks.
Chunk #1 preview:
 # NOVASPHERE DYNAMICS - EMPLOYEE HANDBOOK & OPERATIONAL POLICIES
Document Version: 2026.3
Last Revised: September 15, 2026

1. WELCOME & CORE MISSION
Novasphere Dynamics builds intelligent, quantum-inspired orchestration systems for autonomous industrial robotics and real-time sensor fleets. Our mission is to eliminate latency and operational failures in mission-critical environments.

Chunk #2 preview:
 2. WORKPLACE POLICIES & REMOTE FLEXIBILITY
- Work Locations: Novasphere is a remote-first organization. Employees may work from any location within +/- 4 time zones of their designated hub (Zurich, Neo-Tokyo, or San Francisco).
- Core Collaboration Hours: All team members must be available on Slack and video during core collaboration hours: 13:00 to 16:00 Central European Time (CET).


> **What just happened?**
> Notice that the splitter cleanly kept headings and paragraphs together where possible.

### Step 2: Experiment: The Impact of Chunk Size
Let's run an experiment table comparing chunk sizes of 100, 300, and 800 characters on the exact same document.

In [3]:
results = []
for size in [100, 300, 800]:
    s = RecursiveCharacterTextSplitter(chunk_size=size, chunk_overlap=20)
    c = s.split_documents([handbook_doc])
    lengths = [len(x.page_content) for x in c]
    results.append({
        "chunk_size": size,
        "num_chunks": len(c),
        "min_len": min(lengths),
        "max_len": max(lengths),
        "avg_len": sum(lengths) // len(lengths)
    })

print(f"{'Chunk Size':<12} | {'Num Chunks':<12} | {'Min Len':<10} | {'Max Len':<10} | {'Avg Len':<10}")
print("-" * 65)
for r in results:
    print(f"{r['chunk_size']:<12} | {r['num_chunks']:<12} | {r['min_len']:<10} | {r['max_len']:<10} | {r['avg_len']:<10}")

Chunk Size   | Num Chunks   | Min Len    | Max Len    | Avg Len   
-----------------------------------------------------------------
100          | 62           | 21         | 99         | 72        
300          | 21           | 122        | 278        | 196       
800          | 7            | 220        | 795        | 592       


> **What just happened?**
> Smaller chunk size produces many small snippets; larger chunk size produces fewer, broader chunks.

### Step 3: The Failure Case: A Bad Split That Cuts an Answer in Half
Let's demonstrate a critical failure: split with `chunk_size=100` and `chunk_overlap=0` on the travel expense policy.

In [4]:
fragile_splitter = RecursiveCharacterTextSplitter(chunk_size=100, chunk_overlap=0)
bad_chunks = fragile_splitter.split_documents([handbook_doc])

target_phrase = "$150 per day"
found_chunks = [c for c in bad_chunks if target_phrase in c.page_content]

print(f"Found {len(found_chunks)} chunk containing '{target_phrase}':")
for c in found_chunks:
    print(">>>", repr(c.page_content))

Found 1 chunk containing '$150 per day':
>>> '- Meal Expense Limit: Employees traveling on company business are reimbursed up to $150 per day for'


> **What just happened?**
> Look at the chunk: it ends right after '$150 per day for meals'! The critical condition 'without requiring prior VP approval' was cut off into the next chunk!

### Step 4: The Overlap Fix: Preserving Boundary Context
Now let's add `chunk_overlap=45`. Notice how the sliding window retains the complete policy in both chunks.

In [5]:
safe_splitter = RecursiveCharacterTextSplitter(chunk_size=120, chunk_overlap=45)
fixed_chunks = safe_splitter.split_documents([handbook_doc])

fixed_targets = [c for c in fixed_chunks if target_phrase in c.page_content]
print(f"With overlap=45, found {len(fixed_targets)} chunk(s):")
for idx, c in enumerate(fixed_targets, 1):
    print(f"\n--- Safe Chunk #{idx} ---")
    print(c.page_content)

With overlap=45, found 2 chunk(s):

--- Safe Chunk #1 ---
- Meal Expense Limit: Employees traveling on company business are reimbursed up to $150 per day for meals without

--- Safe Chunk #2 ---
up to $150 per day for meals without requiring prior VP approval. Single meals exceeding $150 require written approval


> **What just happened?**
> Both the dollar amount AND the VP approval rule are preserved together. Overlap saved the answer!

### Step 5: Preserving Metadata Across Chunks
When a document is split into 10 chunks, does the metadata survive? Let's verify.

In [6]:
for idx, chunk in enumerate(chunks[:3]):
    chunk.metadata["chunk_index"] = idx
    print(f"Chunk #{idx} Metadata: {chunk.metadata}")

Chunk #0 Metadata: {'source': 'C:\\project 03\\llm\\course\\data\\company_handbook.txt', 'chunk_index': 0}
Chunk #1 Metadata: {'source': 'C:\\project 03\\llm\\course\\data\\company_handbook.txt', 'chunk_index': 1}
Chunk #2 Metadata: {'source': 'C:\\project 03\\llm\\course\\data\\company_handbook.txt', 'chunk_index': 2}


> **What just happened?**
> Every chunk automatically inherited the source metadata from the parent document.

## 4. Try It Yourself!

### Exercises
1. **Exercise 1 (Markdown Splitting):** Load `product_faq.md` and split it using `chunk_size=300, chunk_overlap=30`. Print the total number of chunks.
2. **Exercise 2 (Inspect Overlap):** Write a function `print_overlap(chunk_a, chunk_b)` that finds and prints the overlapping text string between two consecutive chunks.

In [7]:
faq_doc = TextLoader(str(data_dir / "product_faq.md"), encoding="utf-8").load()[0]
faq_chunks = RecursiveCharacterTextSplitter(chunk_size=300, chunk_overlap=30).split_documents([faq_doc])
print(f"Split FAQ into {len(faq_chunks)} chunks.")

def print_overlap(a: str, b: str):
    for i in range(len(a), 0, -1):
        suffix = a[-i:]
        if b.startswith(suffix):
            print(f"Shared overlap ({len(suffix)} chars): {repr(suffix)}")
            return
    print("No direct boundary overlap found.")

print_overlap(chunks[0].page_content, chunks[1].page_content)

Split FAQ into 15 chunks.
No direct boundary overlap found.


## 5. Common Mistakes & Pitfalls
- **Zero Overlap:** Setting `chunk_overlap=0` will inevitably cut key sentences in half at arbitrary character limits.
- **Excessive Overlap:** Setting overlap to 50%+ of chunk size creates redundant chunks and bloats vector store costs.
- **Character Count vs Token Count:** LLMs measure limits in tokens (~4 characters per token in English). Keep this in mind when sizing chunks.
- **Splitting Code or Markdown Tables Naively:** Naive splitters can break table rows. Use larger chunks or specialized Markdown splitters for structured tables.

---

## 6. Notebook Recap
- Chunking balances search precision against context completeness.
- RecursiveCharacterTextSplitter respects natural paragraphs and sentences before character splits.
- Chunk overlap (typically 10-20% of chunk size) prevents facts at boundaries from being broken.
- Metadata is preserved across all child chunks.


## 7. What's Next?

In **04_embeddings_and_vector_stores.ipynb**, we turn these text chunks into mathematical vectors of meaning, compute cosine similarity by hand, and build our first vector database.

## 8. Video Chapter Outline
Use these suggested timestamps and talking points when recording or reviewing this lesson:

- **00:00 - The Purpose of Chunking:** Why we cannot search entire documents at once.
- **02:20 - Understanding Recursive Splitting:** The paragraph -> sentence -> word hierarchy.
- **05:10 - The Chunk Size Experiment:** Comparing sizes 100, 300, and 800.
- **08:15 - The Boundary Severance Bug:** How bad chunking cuts facts in half.
- **11:00 - The Overlap Remedy:** Sliding windows and metadata inheritance.
